In [8]:
import pandas as pd
import requests
import pathlib
import time


DOWNLOAD_URL_TEMPLATE = "https://miplib.zib.de/WebData/instances/%s.mps.gz"
DOWNLOAD_FILE_TEMPLATE = "%s.mps.gz"

In [11]:
def load_non_continuous(
    csv_info_filename,
    min_vars,
    max_vars,
    min_int_vars,
    max_int_vars,
    min_cont_vars,
    max_cont_vars,
    min_bin_vars,
    max_bin_vars,
    tags
):
    # read table
    df = pd.read_csv(csv_info_filename)
    filtrated = df[df["Continuous  Con."] == 0]

    # binary only
    # filtrated = filtrated[filtrated["Integers  Int."] == 0]
    
    filtrated = filtrated[filtrated["Objective  Obj."].str.contains(r'\d[.]\d') == False]
    if max_vars > 0:
        filtrated = filtrated[filtrated["Variables  Var."] <= max_vars]
    if min_vars > 0:
        filtrated = filtrated[filtrated["Variables  Var."] >= min_vars]
    if max_int_vars > 0:
        filtrated = filtrated[filtrated["Integers  Int."] <= max_int_vars]
    if min_int_vars > 0:
        filtrated = filtrated[filtrated["Integers  Int."] >= min_int_vars]
    if max_cont_vars > 0:
        filtrated = filtrated[filtrated["Continuous  Con."] <= max_cont_vars]
    if min_cont_vars > 0:
        filtrated = filtrated[filtrated["Continuous  Con."] >= min_cont_vars]
    if max_bin_vars > 0:
        filtrated = filtrated[filtrated["Binaries  Bin."] <= max_bin_vars]
    if min_bin_vars > 0:
        filtrated = filtrated[filtrated["Binaries  Bin."] >= min_bin_vars]
    if tags != "":
        prepared_tags = tags.replace(",", "|")
        print(prepared_tags)
        filtrated = filtrated[filtrated["Tags  Tag."].str.contains(prepared_tags) == True]
    print("*****")
    print(filtrated)
    return filtrated

In [16]:
def download_benchmarks(
    csv_info_filename,
    output_folder,
    min_vars,
    max_vars,
    min_int_vars,
    max_int_vars,
    min_cont_vars,
    max_cont_vars,
    min_bin_vars,
    max_bin_vars,
    tags
):
    output_folder = output_folder if output_folder[-1] == '/' else output_folder + '/'
    pathlib.Path(output_folder).mkdir(parents=True, exist_ok=True)

    df = load_non_continuous(
        csv_info_filename,
        min_vars,
        max_vars,
        min_int_vars,
        max_int_vars,
        min_cont_vars,
        max_cont_vars,
        min_bin_vars,
        max_bin_vars,
        tags
    )
    size = len(df.index)
    print(f"found {size} benchmarks")
    for i, row in df.iterrows():
        url = DOWNLOAD_URL_TEMPLATE % row["Instance  Ins."]
        save_path = output_folder + DOWNLOAD_FILE_TEMPLATE % f'{row["Instance  Ins."]}_vc{row["Variables  Var."]}'
        response = requests.get(url, stream=True)

        with open(save_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024):
                file.write(chunk)

        if i % 10 == 0:
            print(f"progress: {i}/{size}")
            time.sleep(2)
        else:
            time.sleep(1)

    print("done!")

In [15]:
csv_info_filename = "The Benchmark Set.csv"
output_folder = "../benchmarks/"
min_vars = 0
max_vars = 10000
min_int_vars = 0
max_int_vars = 10000
min_cont_vars = 0
max_cont_vars = 10000
min_bin_vars = 0
max_bin_vars = 10000
tags = "aggregations"


download_benchmarks(
    csv_info_filename,
    output_folder,
    min_vars,
    max_vars,
    min_int_vars,
    max_int_vars,
    min_cont_vars,
    max_cont_vars,
    min_bin_vars,
    max_bin_vars,
    tags
)

aggregations
*****
       Instance  Ins. Status  Sta.  Variables  Var.  Binaries  Bin.   
12     hypothyroid-k1         easy             2602            2601  \
23   peg-solitaire-a3         easy             4552            4552   
24        neos-950242         easy             5760            5520   
28      supportcase22         hard             7129            7129   
31  neos-3083819-nubu         easy             8644               0   

    Integers  Int.  Continuous  Con.  Constraints  Con.  Nonz.  Non.   
12               1                 0               5195       433884  \
23               0                 0               4587        28387   
24             240                 0              34224       104160   
28               0                 0             260602      2228828   
31            8644                 0               4725        24048   

                 Submitter  Sub.                Group  Gro. Objective  Obj.   
12                    Gleb Belov          